# 10 · 병렬 알고리즘 cuda-cccl (`cuda.compute`)

> **CuPy 2일 집중 코스 — Day 2 / 단원 7 (병렬 알고리즘, GTC 05 기반)**

08·09에서 커널을 **직접** 짰다면, 여기서는 NVIDIA가 제공하는 **검증된 병렬 알고리즘**으로
같은 일을 더 쉽고 빠르게 합니다. `cuda.compute`(cuda-cccl)의 reduce·scan·transform·sort 등은
손으로 튜닝한 CUDA 커널 수준의 성능을 Python에서 제공합니다.

## 이 노트북의 위치 (07 개념과의 관계)
- 07~09의 개념(인덱싱·coalescing·공유메모리·atomic)을 **알고리즘이 내부에서 처리** → 우리는 *무엇을* 할지만 지정
- 직접 커널(생산성↓·성능↑·제어↑) vs cccl(생산성↑·검증된 성능) 의 트레이드오프

## 학습 목표
- `reduce_into`로 합/커스텀 리덕션을 수행한다.
- 사용자 정의 이항연산·`unary_transform`·iterator를 활용한다.
- 언제 cccl / Numba / CuPy를 쓸지 판단한다.

## 목차
1. [cuda-cccl 이란 & 설치](#1)
2. [reduce_into — 합](#2)
3. [커스텀 이항연산 리덕션](#3)
4. [unary_transform](#4)
5. [Iterators (메모리 없는 시퀀스)](#5)
6. [언제 무엇을 쓰나](#6)
7. [스캔·커스텀·평가](#8) · 8. [연습](#7-ex)

> `cuda-cccl`은 **실험적 패키지**(2026 기준 `cuda.compute`, v0.5.x)입니다. API가 버전에 따라 바뀔 수 있습니다.

<a id="1"></a>
## 1. cuda-cccl 이란 & 설치

**CCCL**(CUDA Core Compute Libraries)의 Python 인터페이스 `cuda.compute` 는 배열/범위 단위 **병렬 알고리즘**을 제공합니다.
- reduce, scan, sort, transform, … (아키텍처 이식성 + 고성능)
- 사용자 정의 연산은 내부적으로 **numba-cuda**로 컴파일됩니다(람다 불가, 클로저 권장).

설치(강의장 환경): `pip install "cuda-python[cu13]" "cuda-cccl[cu13]"`. 미설치 시 아래 셀은 안내만 출력합니다.

In [ ]:
import numpy as np, cupy as cp
from course_utils import print_env, allclose
print_env()
try:
    import cuda.compute
    from cuda.compute import OpKind, CountingIterator, TransformIterator
    HAS_CCCL = True
    print('cuda.compute 사용 가능')
except Exception as e:
    HAS_CCCL = False
    print('cuda-cccl 미설치 — `pip install cuda-cccl` 후 실행하세요. (',e,')')

<a id="2"></a>
## 2. reduce_into — 합

`reduce_into(d_in, d_out, op, num_items, h_init)`. `op`은 내장 `OpKind.PLUS` 등.
`h_init`은 **host(numpy) 초기값 배열**, `d_out`은 결과를 담을 device 배열.

In [ ]:
dtype = np.int64
d_in  = cp.arange(1, 1_000_001, dtype=dtype)   # 1..1,000,000
d_out = cp.empty(1, dtype=dtype)
h_init = np.array([0], dtype=dtype)

cuda.compute.reduce_into(
    d_in=d_in, 
    d_out=d_out, 
    op=OpKind.PLUS, 
    num_items=d_in.size, 
    h_init=h_init
)

print('cccl 합:', int(d_out[0]), '/ cupy 합:', int(d_in.sum()))
allclose(int(d_in.sum()), int(d_out[0]), name='reduce sum')

<a id="3"></a>
## 3. 커스텀 이항연산 리덕션

내장 연산 대신 **함수**를 넘기면 임의의 리덕션이 됩니다(람다 불가). 예: 짝수만 합산.

In [ ]:
def add_even(a, b):
    return (a if a % 2 == 0 else 0) + (b if b % 2 == 0 else 0)

d_in = cp.array([1, 2, 3, 4, 5, 6], dtype=np.int32)
d_out = cp.empty(1, dtype=np.int32)
h_init = np.array([0], dtype=np.int32)

cuda.compute.reduce_into(
    d_in=d_in, 
    d_out=d_out, 
    op=add_even,           # 우리가 만든 커스텀 함수를 전달
    num_items=d_in.size, 
    h_init=h_init
)

print('짝수 합:', int(d_out[0]))   # 2+4+6 = 12


<a id="4"></a>
## 4. unary_transform

각 원소에 함수를 적용해 출력 배열에 씁니다: `unary_transform(d_in, d_out, func, num_items)`.

In [ ]:
def sq(x): 
    return x * x

d_in = cp.arange(10, dtype=np.int32)
d_out = cp.empty(10, dtype=np.int32)

cuda.compute.unary_transform(
    d_in=d_in, 
    d_out=d_out, 
    op=sq, 
    num_items=d_in.size
)

print(cp.asnumpy(d_out))   # [ 0  1  4  9 16 25 36 49 64 81]

<a id="5"></a>
## 5. Iterators — 메모리 없는 시퀀스

`CountingIterator`/`TransformIterator`는 **메모리를 할당하지 않고** 시퀀스를 표현해 알고리즘에 바로 넣습니다.
예: 10부터의 정수 100개의 합(배열 생성 없이).

In [ ]:
# 시작값을 10으로 하는 가상 이터레이터 생성
first = CountingIterator(np.int32(10)) 
# 이 기능의 진정한 매력은 GPU 메모리를 단 1바이트도 쓰지 않는다는 점입니다.
# cp.arange로 배열을 만들면 GPU VRAM을 차지하지만, CountingIterator는 숫자가 필요할 때마다 GPU 코어가 즉석에서 계산해 내므로 대용량 데이터를 처리할 때 메모리 절약 효과가 큼
d_out = cp.empty(1, dtype=np.int32)

cuda.compute.reduce_into(
    d_in=first, 
    d_out=d_out, 
    op=OpKind.PLUS, 
    num_items=100, 
    h_init=np.array([0], dtype=np.int32)
)

# 10부터 109까지 총 100개 숫자의 합 계산 (정답: 5950)
print('합(10..109):', int(d_out[0]))

<a id="6"></a>
## 6. 언제 무엇을 쓰나

| 선택 | 쓸 때 |
|------|-------|
| **CuPy 함수**(`cp.sum`…) | 표준 연산, 가장 간단 |
| **`@cupy.fuse`/Elementwise/Reduction** | 원소·리덕션 융합, 약간의 커스텀 |
| **cuda-cccl**(`cuda.compute`) | reduce/scan/sort/transform + **커스텀 연산**을 검증된 성능으로 |
| **Numba CUDA** | 공유메모리·atomic·복잡한 인덱싱 등 **완전한 제어** |

정리: 표준은 CuPy, 정형 병렬패턴은 cccl, 특수 커널은 Numba.

<a id="7"></a>
## 7. 스캔 · 커스텀 타입 · 종합 평가

리덕션 외에 **스캔(prefix)**, **커스텀 구조체 타입** 평가 과제를 다룹니다.
> 스캔 API는 실험적이라 버전에 따라 인자 순서가 다를 수 있습니다 — 공식 예제로 확인하세요.

### 7.1 스캔(prefix sum)

**exclusive scan**: `out[i] = op(in[0..i-1])` (자기 앞까지 누적). 누적합·러닝 통계의 기본.
-  Exclusive Scan (배타적 누적합)
 * 배열이 온통 1로 채워져 있는데 결과가 0, 1, 2, 3...으로 나오는 이유는 자기 자신을 포함하지 않고(Exclusive) 이전까지의 값들만 더하기 때문입니다.
 * 인덱스 0: 이전 값이 없으므로 초기값인 0
 * 인덱스 1: 이전 값(인덱스 0의 1) 하나만 있으므로 0 + 1 = 1
 * 인덱스 2: 이전 값(인덱스 0, 1의 1) 두 개가 있으므로 0 + 1 + 1 = 2
 * 병렬 컴퓨팅에서 각 스레드가 메모리 어디에 데이터를 써야 할지 오프셋(인덱스 시작 위치)을 계산할 때 사용되는 알고리즘

In [ ]:
# 러닝 합 (exclusive scan, PLUS)
N = 10
d_in = cp.ones(N, dtype=np.int32)
d_out = cp.empty(N, dtype=np.int32)
h_init = np.array([0], dtype=np.int32)

cuda.compute.exclusive_scan(
    d_in=d_in, 
    d_out=d_out, 
    op=OpKind.PLUS, 
    init_value=h_init,
    num_items=N
)
print(cp.asnumpy(d_out))   # [0 1 2 3 4 5 6 7 8 9]


### 7.2 커스텀 구조체 타입 (`gpu_struct`)

사용자 정의 타입으로도 리덕션이 됩니다. 예: 픽셀들 중 **녹색(g)이 최대**인 픽셀 찾기(공식 예제).

In [ ]:
import numpy as np
import cupy as cp
from cuda.compute import gpu_struct, OpKind

@gpu_struct
class Pixel:
    r: np.int32
    g: np.int32
    b: np.int32

def max_g(x, y): 
    return x if x.g > y.g else y

# (10, 3) 모양의 RGB 난수 배열을 생성한 뒤, Pixel 구조체 배열로 뷰(View) 변환
d_rgb = cp.random.randint(0, 256, (10, 3), dtype=np.int32).view(Pixel.dtype)
d_out = cp.empty(1, dtype=Pixel.dtype)
h_init = np.zeros(1, dtype=Pixel.dtype)

cuda.compute.reduce_into(
    d_in=d_rgb, 
    d_out=d_out, 
    op=max_g, 
    num_items=d_rgb.size, 
    h_init=h_init
)

print('전체 배열 (원본):\n', d_rgb)
print('가장 초록색이 강한 픽셀:', d_out.get()[0])

### 7.3 종합 평가 — 센서 '직전 최고치' (ref: GTC CUDA python)

센서 온도 스트림에서 **각 시점 직전까지의 최고 온도**를 구하세요 — `max` 연산 **exclusive scan**으로.
`prev_peak[i] = max(readings[0..i-1])` (i=0은 초기값).

In [ ]:
import numpy as np
import cupy as cp
import cuda.compute

# 커스텀 최댓값 연산 함수
def max_op(a, b): 
    return a if a > b else b

N = 1_000_000
readings = cp.random.randint(0, 100, size=N, dtype=np.int32)
prev_peak = cp.empty(N, dtype=np.int32)

# 1. 매우 작은 초기값 설정 (-2^31)
h_init = np.array([-2147483648], dtype=np.int32)

# 2. CCCL exclusive_scan 호출
cuda.compute.exclusive_scan(
    d_in=readings, 
    d_out=prev_peak, 
    op=max_op, 
    init_value=h_init,  
    num_items=N
)

# --- 3. 검증 (Validation - NumPy 사용) ---
# CuPy 대신 NumPy로 데이터를 가져와서 검증합니다.
# CuPy에서는 아직 maximum.accumulate 기능을 구현해 놓지 않았습니다.
readings_np = readings.get()

expected_np = np.empty(N, dtype=np.int32)
expected_np[0] = h_init[0]
expected_np[1:] = np.maximum.accumulate(readings_np[:-1])

# 작은 N(앞부분 10개) 결과 눈으로 확인하기
print("입력 데이터 :", readings_np[:10])
print("CCCL 결과   :", prev_peak[:10].get())
print("NumPy(기대값):", expected_np[:10])

# 전체 100만 개 데이터 일치 여부 검증
np.testing.assert_array_equal(prev_peak.get(), expected_np)
print("\n[전체 데이터 검증 완료] CCCL exclusive_scan == NumPy accumulate")

<a id="7"></a>
## 8. 연습 — 커스텀 op 최댓값

**연습 — 커스텀 op로 최댓값**: `reduce_into`에 `max_op(a,b)`와 적절한 `h_init`(아주 작은 값)을 줘서 최댓값을 구하세요.

In [ ]:

import numpy as np
import cupy as cp
import cuda.compute

# from course_utils import allclose

def max_op(a, b):
    return a if a > b else b

# 100만 개의 무작위 정수 배열 생성 (0 ~ 999,999)
d_in = cp.random.randint(0, 10**6, size=1_000_000, dtype=np.int32)
d_out = cp.empty(1, dtype=np.int32)

# np.iinfo(np.int32).min 을 사용하여 int32의 가장 작은 값(-2147483648)을 동적으로 가져옵니다.
# 이전 예제에서는 -2147483648을 직접 쳤지만, 실무에서는 데이터 타입이 수시로 바뀔 수 있습니다. 
# np.iinfo(np.int32).min을 사용하면 데이터 타입에 맞는 최소값을 알아서 찾아주기 때문에 보다 안전한 코드입니다.
h_init = np.array([np.iinfo(np.int32).min], dtype=np.int32)

# reduce_into 호출
cuda.compute.reduce_into(
    d_in=d_in,
    d_out=d_out,
    op=max_op,
    num_items=d_in.size,
    h_init=h_init
)

# CuPy 내장 함수인 .max()와 결과 비교하기
print('CCCL 최댓값:', int(d_out[0]), '/ CuPy 최댓값:', int(d_in.max()))

# 검증 (두 값이 일치하면 'allclose OK' 출력)
allclose(int(d_in.max()), int(d_out[0]), name='reduce max')

### 체크포인트
- [ ] `reduce_into`로 합·커스텀 리덕션을 수행했다
- [ ] `unary_transform`·iterator를 사용했다
- [ ] cccl / Numba / CuPy의 사용 시점을 구분한다
- [ ] (참고) scan/sort 등 다른 알고리즘도 같은 방식으로 쓸 수 있음을 안다

다음: **`11_rawkernel`** — RawKernel(CUDA C)과 커널 개념 적용(최적화) 기술.